# 04 Limpieza 08 — Deduplicar y fusionar artículos/autores UNAM

Esta libreta deduplica y fusiona la base ya normalizada de autores.

Criterio principal: **título normalizado + año + autor normalizado**. El DOI **no** se usa como llave de unión; sólo se conserva/fusiona como metadato bibliográfico.

In [1]:
from __future__ import annotations

import csv
import json
import re
import unicodedata
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

## Rutas del proyecto

## Ruta real del proyecto en Windows.

In [2]:
# ---------------------------------------------------------------------
# Rutas del proyecto
# ---------------------------------------------------------------------
INPUT_CSV = Path(r"C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\03_Nomralizar_Nombres\integrado_solo_unam_autores_normalizados.csv")
OUTPUT_DIR = Path(r"C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\04_deduplicar")

if not INPUT_CSV.exists() and Path("/mnt/data/integrado_solo_unam_autores_normalizados.csv").exists():
    INPUT_CSV = Path("/mnt/data/integrado_solo_unam_autores_normalizados.csv")
    OUTPUT_DIR = Path("/mnt/data/04_deduplicar_pares_revisados_final")

OUTPUT_CSV = OUTPUT_DIR / "integrado_solo_unam_deduplicado_fusionado.csv"
OUTPUT_ARTICLES_CSV = OUTPUT_DIR / "integrado_solo_unam_deduplicado_fusionado_articulos.csv"
DIAG_ARTICLES_CSV = OUTPUT_DIR / "diagnostico_deduplicacion_fusion_articulos.csv"
DIAG_AUTHORS_CSV = OUTPUT_DIR / "diagnostico_deduplicacion_fusion_autores.csv"
SUMMARY_JSON = OUTPUT_DIR / "resumen_deduplicacion_fusion.json"

CANONICAL_COLUMNS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract",
]

ARTICLE_DIAG_COLUMNS = [
    "indice_fusionado", "indices_originales", "n_indices_originales", "n_filas_originales",
    "n_filas_finales_autor", "reglas_deduplicacion", "clave_principal_usada",
    "Titulo_final", "titulos_originales", "Año_final", "años_originales",
    "autores_interseccion_usados", "Doi_final", "dois_originales", "ISBN_final",
    "isbn_originales", "ISSN_final", "issn_originales", "URL_final", "urls_originales",
    "Area_final", "areas_originales", "Keywords_final", "Abstract_longitud_final",
    "n_autores_finales", "autores_finales", "conflicto_titulo", "conflicto_año",
    "conflicto_area", "conflicto_doi", "motivo_revision",
]

AUTHOR_DIAG_COLUMNS = [
    "indice_fusionado", "Autor_norm_final", "autor_key", "indices_originales",
    "n_filas_originales_autor", "variantes_autor_originales", "Afiliacion1_final",
    "Afiliacion2_final", "afiliaciones_candidatas", "afiliaciones_descartadas",
    "motivo_fusion_autor", "Titulo_final", "Doi_final",
]

ARTICLE_LEVEL_COLUMNS = [
    "indice", "Titulo", "Año", "Autores_UNAM", "Afiliaciones_UNAM", "ISBN", "ISSN",
    "Doi", "URL", "Area", "Keywords", "Abstract", "indices_originales", "n_autores_unam",
]

FALSE_EMPTY = {
    "", "na", "n/a", "nan", "none", "null", "n d", "n/d", "-", "--",
    "sin informacion", "sin información", "no disponible", "not available",
}

GENERIC_UNAM = "Universidad Nacional Autonoma de Mexico"
GENERIC_UNAM_KEYS = {
    "universidad nacional autonoma de mexico",
    "universidad nacional autónoma de méxico",
}

VALID_AREAS = ["ISBD", "CC", "IA", "TC", "SIAV", "RS"]
AREA_PRIORITY = {area: i for i, area in enumerate(VALID_AREAS)}

## Utilidades de texto

In [3]:
# ---------------------------------------------------------------------
# Utilidades de texto
# ---------------------------------------------------------------------
def strip_accents(text: str) -> str:
    text = "" if text is None else str(text)
    return "".join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )


def norm_text(text: str) -> str:
    text = strip_accents("" if text is None else str(text)).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_cell(value: Any) -> str:
    value = "" if value is None else str(value)
    value = value.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    value = re.sub(r"\s+", " ", value).strip()
    if norm_text(value) in FALSE_EMPTY:
        return ""
    return value


def clean_title_output(title: str) -> str:
    title = clean_cell(title)
    if not title:
        return ""
    title = re.sub(r"^hot\s+off\s+the\s+press\s*:\s*", "", title, flags=re.I)
    title = re.sub(r"\s*\[Formula presented\]\s*", "", title, flags=re.I)
    title = re.sub(r"\bNeuralode\b", "NeuralODE", title, flags=re.I)
    title = re.sub(r"\bT2Spectrum\b", "T2 Spectrum", title)
    title = re.sub(r"\bHateSpeech\b", "Hate Speech", title)
    title = re.sub(r"\bSpanishspeaking\b", "Spanish speaking", title)
    title = re.sub(r"\bR\s+2\b", "R2", title)
    title = re.sub(r"a\)\.?$", "", title)
    title = re.sub(r"\breconstructiona\.?$", "reconstruction", title, flags=re.I)
    return re.sub(r"\s+", " ", title).strip(" ;")


def title_key(title: str) -> str:
    title = clean_title_output(title)
    title = re.sub(r"\[[^\]]*\]", " ", title)
    title = strip_accents(title).lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    return re.sub(r"\s+", " ", title).strip()


def author_key(author: str) -> str:
    return norm_text(author)


def norm_doi(value: str) -> str:
    value = clean_cell(value).lower()
    value = re.sub(r"^(?:https?://(?:dx\.)?doi\.org/|doi\.org/|doi:|doi/)", "", value)
    value = value.strip().strip(".").strip("/")
    if value.startswith("10."):
        return value
    return ""


def split_comma_values(value: str) -> List[str]:
    value = clean_cell(value)
    if not value:
        return []
    return [clean_cell(x) for x in re.split(r"\s*,\s*", value) if clean_cell(x)]


def split_keywords(value: str) -> List[str]:
    value = clean_cell(value).lower()
    if not value:
        return []
    value = value.replace(";", ",").replace("|", ",")
    return [clean_cell(p).lower() for p in re.split(r"\s*,\s*", value) if clean_cell(p)]


def unique_preserve(values: Iterable[str], key_func=norm_text) -> List[str]:
    out: List[str] = []
    seen = set()
    for value in values:
        value = clean_cell(value)
        if not value:
            continue
        key = key_func(value)
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(value)
    return out

## Selección/fusión de metadatos

In [4]:
# ---------------------------------------------------------------------
# Selección/fusión de metadatos
# ---------------------------------------------------------------------
def title_badness(title: str) -> int:
    t = clean_cell(title).lower()
    bad = 0
    if re.search(r"^hot\s+off\s+the\s+press\s*:", t):
        bad += 100
    if "[formula presented]" in t:
        bad += 80
    if "; [" in t:
        bad += 40
    if re.search(r"a\)\.?$", t):
        bad += 30
    if re.search(r"\b(hatespeech|spanishspeaking|t2spectrum|neuralode)\b", t):
        bad += 15
    return bad


def choose_title(values: List[str]) -> str:
    vals = [v for v in values if clean_cell(v)]
    if not vals:
        return ""
    freq = Counter(vals)
    ranked = []
    for value in vals:
        cleaned = clean_title_output(value)
        if cleaned:
            ranked.append((title_badness(value), -freq[value], -len(cleaned), cleaned))
    ranked.sort()
    return ranked[0][3] if ranked else ""


def choose_year(values: List[str]) -> str:
    vals = [clean_cell(v) for v in values if clean_cell(v)]
    if not vals:
        return ""
    return Counter(vals).most_common(1)[0][0]


def merge_identifier_values(values: List[str]) -> str:
    parts: List[str] = []
    for value in values:
        parts.extend(split_comma_values(value))
    return ", ".join(unique_preserve(parts, key_func=lambda x: norm_text(x)))


def choose_doi(values: List[str]) -> str:
    dois = [norm_doi(v) for v in values if norm_doi(v)]
    if not dois:
        return ""
    counts = Counter(dois)
    def score(doi: str) -> Tuple[int, int, int, str]:
        # DOI editorial antes que arXiv si hay conflicto.
        is_arxiv = 1 if "10.48550/arxiv" in doi else 0
        return (counts[doi], -is_arxiv, len(doi), doi)
    return sorted(counts, key=score, reverse=True)[0]


def choose_url(values: List[str], doi: str) -> str:
    if doi:
        return f"https://doi.org/{doi}"
    vals = unique_preserve([clean_cell(v) for v in values if clean_cell(v)], key_func=lambda x: x.strip())
    vals = [v for v in vals if v.startswith("http")]
    if not vals:
        return ""
    return sorted(vals, key=lambda x: (x.startswith("https://"), len(x)), reverse=True)[0]


def merge_keywords(values: List[str]) -> str:
    parts: List[str] = []
    for value in values:
        parts.extend(split_keywords(value))
    return ", ".join(unique_preserve(parts, key_func=norm_text))


def choose_abstract(values: List[str]) -> str:
    vals = unique_preserve([clean_cell(v) for v in values if clean_cell(v)], key_func=norm_text)
    if not vals:
        return ""
    # Más completo: más largo. En empate, más contenido alfabético.
    def score(v: str) -> Tuple[int, int, str]:
        alpha = sum(ch.isalpha() for ch in v)
        return (len(v), alpha, v)
    return sorted(vals, key=score, reverse=True)[0]


def split_area_values(value: str) -> List[str]:
    areas = []
    for part in split_comma_values(value):
        p = clean_cell(part).upper()
        if p in VALID_AREAS:
            areas.append(p)
    return areas


def choose_area(values: List[str]) -> str:
    parts: List[str] = []
    for value in values:
        parts.extend(split_area_values(value))
    if not parts:
        return ""
    counts = Counter(parts)
    first_pos = {a: parts.index(a) for a in set(parts)}
    return sorted(counts, key=lambda a: (counts[a], -first_pos[a], -AREA_PRIORITY.get(a, 999)), reverse=True)[0]


def has_accent(text: str) -> bool:
    return any(unicodedata.category(c).startswith("M") for c in unicodedata.normalize("NFD", text))


def choose_author_display(values: List[str]) -> str:
    vals = [clean_cell(v) for v in values if clean_cell(v)]
    if not vals:
        return ""
    counts = Counter(vals)
    def score(v: str) -> Tuple[int, int, int, int, int, str]:
        toks = [t for t in re.split(r"\s+", v) if t]
        full = [t for t in toks if len(norm_text(t)) > 1]
        initials = [t for t in toks if len(norm_text(t)) == 1]
        return (len(full), -len(initials), int(has_accent(v)), len(v), counts[v], v)
    return sorted(counts, key=score, reverse=True)[0]

## Afiliaciones

In [5]:
# ---------------------------------------------------------------------
# Afiliaciones
# ---------------------------------------------------------------------
def affiliation_key(value: str) -> str:
    return norm_text(value)


def is_generic_unam(value: str) -> bool:
    return affiliation_key(value) in GENERIC_UNAM_KEYS


def choose_affiliations(values: List[str]) -> Tuple[str, str, List[str], List[str], str]:
    vals = unique_preserve(values, key_func=affiliation_key)
    if not vals:
        return "", "", [], [], "sin_afiliacion"

    # Prioridad: si existe una dependencia específica, eliminar Universidad Nacional Autonoma de Mexico.
    specific = [v for v in vals if not is_generic_unam(v)]
    candidates = specific if specific else vals

    counts = Counter(affiliation_key(v) for v in values if clean_cell(v))
    first_pos = {affiliation_key(v): i for i, v in enumerate(vals)}

    def aff_score(v: str) -> Tuple[int, int, int, int]:
        k = affiliation_key(v)
        specificity = 0 if is_generic_unam(v) else 1
        return (specificity, counts[k], len(v), -first_pos.get(k, 999999))

    ordered = sorted(candidates, key=aff_score, reverse=True)
    selected = ordered[:2]
    selected_keys = {affiliation_key(x) for x in selected}
    discarded = [v for v in vals if affiliation_key(v) not in selected_keys]

    aff1 = selected[0] if len(selected) >= 1 else ""
    aff2 = selected[1] if len(selected) >= 2 else ""
    if len(vals) == 1:
        reason = "una_afiliacion"
    elif discarded:
        reason = "afiliaciones_fusionadas_priorizando_dependencia_especifica"
    else:
        reason = "afiliaciones_fusionadas_sin_recorte"
    return aff1, aff2, vals, discarded, reason

## Unión de clusters

In [6]:
# ---------------------------------------------------------------------
# Unión de clusters
# ---------------------------------------------------------------------
class UnionFind:
    def __init__(self, items: Iterable[str]):
        self.parent = {x: x for x in items}
    def find(self, x: str) -> str:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a: str, b: str) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


def natural_idx(x: str) -> Tuple[int, str]:
    return (int(x), x) if str(x).isdigit() else (10**9, str(x))


def read_rows(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != CANONICAL_COLUMNS:
            raise ValueError(f"Columnas no canónicas. Esperadas={CANONICAL_COLUMNS}; recibidas={reader.fieldnames}")
        return [{col: clean_cell(row.get(col, "")) for col in CANONICAL_COLUMNS} for row in reader]


def build_article_records(rows: List[Dict[str, str]]) -> Dict[str, Dict[str, Any]]:
    grouped: Dict[str, List[Tuple[int, Dict[str, str]]]] = defaultdict(list)
    for row_pos, row in enumerate(rows):
        grouped[row["indice"]].append((row_pos, row))

    records: Dict[str, Dict[str, Any]] = {}
    for indice, pairs in grouped.items():
        article_rows = [row for _, row in pairs]
        titles = [r["Titulo"] for r in article_rows if r["Titulo"]]
        years = [r["Año"] for r in article_rows if r["Año"]]
        authors = [r["Autor_norm"] for r in article_rows if r["Autor_norm"]]
        title = choose_title(titles)
        year = choose_year(years)
        records[indice] = {
            "indice": indice,
            "pairs": pairs,
            "title": title,
            "title_key": title_key(title),
            "year": year,
            "author_keys": {author_key(a) for a in authors if author_key(a)},
            "author_names": unique_preserve(authors, key_func=author_key),
        }
    return records


def title_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    # Contención fuerte para títulos con subtítulo/ruido mínimo.
    shorter, longer = (a, b) if len(a) <= len(b) else (b, a)
    if len(shorter) >= 35 and shorter in longer:
        return 0.97
    return SequenceMatcher(None, a, b).ratio()


def build_clusters(records: Dict[str, Dict[str, Any]]) -> Tuple[List[List[str]], Dict[str, List[str]], Dict[str, str]]:
    """
    Construye clusters de artículos duplicados SIN usar DOI como llave de unión.

    Reglas base:
    1) mismo título normalizado + año + autor exacto;
    2) mismo título normalizado + año;
    3) mismo título normalizado + autor compartido aunque el año difiera 2024/2025;
    4) título similar + mismo año + autor compartido;
    5) equivalencias manuales revisadas para cuatro casos puntuales.

    La regla 5 permite fusionar los pares:
    - multi-objective vs multiobjective;
    - ε vs ϵ y multi-objective vs multiobjective;
    siempre que haya compatibilidad de autor por tokens, para evitar sobre-fusión.
    """
    uf = UnionFind(records.keys())
    pair_rules: Dict[Tuple[str, str], List[str]] = defaultdict(list)
    pair_author_overlap: Dict[Tuple[str, str], str] = {}

    def record_author_overlap_exact(a: str, b: str) -> set:
        return records[a]["author_keys"] & records[b]["author_keys"]

    def author_tokens_local(author: str) -> set:
        return {t for t in author_key(author).split() if len(t) >= 2}

    def author_variants_compatible_local(a: str, b: str) -> bool:
        ta, tb = author_tokens_local(a), author_tokens_local(b)
        if not ta or not tb:
            return False
        if author_key(a) == author_key(b):
            return True
        inter = ta & tb
        return len(inter) >= 2 and (ta.issubset(tb) or tb.issubset(ta))

    def compatible_author_overlap(a: str, b: str) -> List[str]:
        """Autor compartido exacto o variante contenida, p. ej. Carlos Hernandez vs Carlos Ignacio Hernández Castellanos."""
        exact = sorted(record_author_overlap_exact(a, b))
        if exact:
            return exact
        out = []
        for name_a in records[a].get("author_names", []):
            for name_b in records[b].get("author_names", []):
                if author_variants_compatible_local(name_a, name_b):
                    # Se guarda la forma más informativa para diagnóstico.
                    out.append(choose_author_display([name_a, name_b]))
        return unique_preserve(out, key_func=author_key)

    def registrar_union(a: str, b: str, regla: str, autores: Iterable[str] | None = None) -> None:
        uf.union(a, b)
        p = tuple(sorted((a, b)))
        pair_rules[p].append(regla)
        autores = list(autores or [])
        if autores:
            pair_author_overlap[p] = " | ".join(unique_preserve(autores, key_func=lambda x: x))

    # -----------------------------------------------------------------
    # 1) Regla principal: mismo título normalizado + año + mismo autor.
    # -----------------------------------------------------------------
    title_year_author_groups: Dict[Tuple[str, str, str], List[str]] = defaultdict(list)
    for idx, rec in records.items():
        if not rec["title_key"] or not rec["year"]:
            continue
        for ak in rec["author_keys"]:
            title_year_author_groups[(rec["title_key"], rec["year"], ak)].append(idx)

    for key, indices in title_year_author_groups.items():
        if len(indices) > 1:
            root = indices[0]
            for idx in indices[1:]:
                registrar_union(root, idx, "mismo_titulo_año_autor", [key[2]])

    # -----------------------------------------------------------------
    # 2) Mismo título normalizado + año.
    # -----------------------------------------------------------------
    title_year_groups: Dict[Tuple[str, str], List[str]] = defaultdict(list)
    for idx, rec in records.items():
        if rec["title_key"] and rec["year"]:
            title_year_groups[(rec["title_key"], rec["year"])].append(idx)

    for key, indices in title_year_groups.items():
        if len(indices) > 1:
            root = indices[0]
            for idx in indices[1:]:
                common = record_author_overlap_exact(root, idx)
                registrar_union(root, idx, "mismo_titulo_normalizado_y_año", sorted(common))

    # -----------------------------------------------------------------
    # 2b) Mismo título normalizado + autor compartido, aunque el año difiera.
    #     Esto corrige los casos 2024 vs 2025 sin usar DOI como llave.
    # -----------------------------------------------------------------
    title_author_cross_year_groups: Dict[Tuple[str, str], List[str]] = defaultdict(list)
    for idx, rec in records.items():
        if not rec["title_key"]:
            continue
        if len(rec["title_key"]) < 45:
            continue
        for ak in rec["author_keys"]:
            title_author_cross_year_groups[(rec["title_key"], ak)].append(idx)

    for (tkey, ak), indices in title_author_cross_year_groups.items():
        if len(indices) <= 1:
            continue
        years = {records[idx]["year"] for idx in indices if records[idx]["year"]}
        if len(years) <= 1:
            continue
        # Este proyecto sólo espera 2024/2025; si aparece otro año, no se fuerza.
        if not years.issubset({"2024", "2025"}):
            continue
        root = indices[0]
        for idx in indices[1:]:
            registrar_union(root, idx, "mismo_titulo_normalizado_autor_compartido_año_distinto", [ak])

    # -----------------------------------------------------------------
    # 3) Título muy parecido + mismo año + autor exacto compartido.
    # -----------------------------------------------------------------
    fuzzy_buckets: Dict[Tuple[str, str], List[str]] = defaultdict(list)
    for idx, rec in records.items():
        if not rec["year"] or not rec["title_key"]:
            continue
        tokens = rec["title_key"].split()
        prefix = " ".join(tokens[:8]) if len(tokens) >= 8 else rec["title_key"]
        fuzzy_buckets[(rec["year"], prefix)].append(idx)

    for _, indices in fuzzy_buckets.items():
        n = len(indices)
        if n < 2:
            continue
        for i in range(n):
            a = indices[i]
            ra = records[a]
            for j in range(i + 1, n):
                b = indices[j]
                rb = records[b]
                common = ra["author_keys"] & rb["author_keys"]
                if not common:
                    continue
                sim = title_similarity(ra["title_key"], rb["title_key"])
                if sim >= 0.965:
                    registrar_union(a, b, f"titulo_similar_año_autor_sim_{sim:.3f}", sorted(common))

    # -----------------------------------------------------------------
    # 4) Equivalencias manuales revisadas.
    #     No usa DOI. Sólo fusiona esos títulos si hay autor compatible.
    # -----------------------------------------------------------------
    def manual_title_family_key(title: str) -> str:
        k = title_key(title)
        # Homologa variantes gráficas/puntuación relevantes.
        k = re.sub(r"\bmulti objective\b", "multiobjective", k)
        k = re.sub(r"\s+", " ", k).strip()

        manual_groups = {
            "solution algorithms for the capacitated location tree problem with interconnections":
                "manual_solution_algorithms_capacitated_location_tree",
            "towards a collaborative game designer using the ccdsf framework":
                "manual_towards_collaborative_game_designer_ccdsf",
            "finding the set of nearly optimal solutions of a multiobjective optimization problem":
                "manual_finding_set_nearly_optimal_multiobjective",
            "finding locally optimal solutions for multiobjective multimodal optimization":
                "manual_finding_epsilon_locally_optimal_multiobjective_multimodal",
        }
        return manual_groups.get(k, "")

    manual_groups: Dict[str, List[str]] = defaultdict(list)
    for idx, rec in records.items():
        mkey = manual_title_family_key(rec.get("title", ""))
        if mkey:
            manual_groups[mkey].append(idx)

    for mkey, indices in manual_groups.items():
        if len(indices) <= 1:
            continue
        for i in range(len(indices)):
            a = indices[i]
            for j in range(i + 1, len(indices)):
                b = indices[j]
                autores = compatible_author_overlap(a, b)
                if not autores:
                    # Al ser regla manual, igual se deja sin fusionar si no hay autor compatible.
                    continue
                registrar_union(a, b, f"fusion_manual_titulo_equivalente_autor_compatible::{mkey}", autores)

    # -----------------------------------------------------------------
    # Construcción final de clusters y diagnóstico.
    # -----------------------------------------------------------------
    clusters_dict: Dict[str, List[str]] = defaultdict(list)
    for idx in records:
        clusters_dict[uf.find(idx)].append(idx)

    clusters = [sorted(indices, key=natural_idx) for indices in clusters_dict.values()]
    clusters.sort(key=lambda group: natural_idx(group[0]))

    cluster_rules: Dict[str, List[str]] = {}
    cluster_authors: Dict[str, str] = {}
    for cluster in clusters:
        rules = set()
        authors_used = set()
        if len(cluster) == 1:
            rules.add("sin_duplicado")
        else:
            for i in range(len(cluster)):
                for j in range(i + 1, len(cluster)):
                    p = tuple(sorted((cluster[i], cluster[j])))
                    for r in pair_rules.get(p, []):
                        rules.add(r)
                    if p in pair_author_overlap:
                        authors_used.add(pair_author_overlap[p])
        key = "||".join(cluster)
        cluster_rules[key] = sorted(rules)
        cluster_authors[key] = " | ".join(sorted(authors_used))
    return clusters, cluster_rules, cluster_authors


def collect_cluster_rows(cluster: List[str], records: Dict[str, Dict[str, Any]]) -> List[Tuple[str, int, Dict[str, str]]]:
    out = []
    for original_idx in cluster:
        for row_pos, row in records[original_idx]["pairs"]:
            out.append((original_idx, row_pos, row))
    out.sort(key=lambda x: x[1])
    return out

## Fusión de cluster

In [7]:
# ---------------------------------------------------------------------
# Fusión de cluster
# ---------------------------------------------------------------------
def aff_keys_from_row(row: Dict[str, str]) -> set:
    out = set()
    for col in ["Afiliacion1", "Afiliacion2"]:
        if row.get(col):
            out.add(affiliation_key(row[col]))
    return out


def author_tokens(author: str) -> set:
    return {t for t in author_key(author).split() if len(t) >= 2}


def author_variants_compatible(a: str, b: str) -> bool:
    ta, tb = author_tokens(a), author_tokens(b)
    if not ta or not tb:
        return False
    if author_key(a) == author_key(b):
        return True
    inter = ta & tb
    return len(inter) >= 2 and (ta.issubset(tb) or tb.issubset(ta))


def fuse_cluster(
    cluster: List[str],
    new_idx: int,
    records: Dict[str, Dict[str, Any]],
    cluster_rules: List[str],
    authors_used_for_merge: str,
) -> Tuple[List[Dict[str, str]], Dict[str, str], List[Dict[str, str]], Dict[str, str]]:
    cluster_rows = collect_cluster_rows(cluster, records)
    rows = [row for _, _, row in cluster_rows]

    all_titles = [r["Titulo"] for r in rows]
    all_years = [r["Año"] for r in rows]
    all_dois = [r["Doi"] for r in rows]
    all_isbn = [r["ISBN"] for r in rows]
    all_issn = [r["ISSN"] for r in rows]
    all_urls = [r["URL"] for r in rows]
    all_areas = [r["Area"] for r in rows]
    all_keywords = [r["Keywords"] for r in rows]
    all_abstracts = [r["Abstract"] for r in rows]

    title_final = choose_title(all_titles)
    year_final = choose_year(all_years)
    doi_final = choose_doi(all_dois)
    isbn_final = merge_identifier_values(all_isbn)
    issn_final = merge_identifier_values(all_issn)
    url_final = choose_url(all_urls, doi_final)
    area_final = choose_area(all_areas)
    keywords_final = merge_keywords(all_keywords)
    abstract_final = choose_abstract(all_abstracts)

    doi_originales = unique_preserve([norm_doi(d) for d in all_dois if norm_doi(d)], key_func=lambda x: x)
    title_originales = unique_preserve(all_titles, key_func=lambda x: norm_text(clean_title_output(x)))
    year_originales = unique_preserve(all_years, key_func=lambda x: x)
    area_originales = unique_preserve([a for v in all_areas for a in split_area_values(v)], key_func=norm_text)
    isbn_originales = unique_preserve([p for v in all_isbn for p in split_comma_values(v)], key_func=norm_text)
    issn_originales = unique_preserve([p for v in all_issn for p in split_comma_values(v)], key_func=norm_text)

    # Autores: fusionar variantes del mismo autor dentro del artículo.
    author_groups: Dict[str, Dict[str, Any]] = {}
    author_order: List[str] = []
    group_seq = 0

    for original_idx, row_pos, row in cluster_rows:
        raw_author = row["Autor_norm"]
        ak = author_key(raw_author)
        if not ak:
            continue
        row_aff_keys = aff_keys_from_row(row)
        target_gid = None
        for gid in author_order:
            group = author_groups[gid]
            if ak in group["exact_keys"]:
                target_gid = gid
                break
            if row_aff_keys and (row_aff_keys & group["aff_keys"]):
                if any(author_variants_compatible(raw_author, existing) for existing in group["variants"]):
                    target_gid = gid
                    group["merge_notes"].append("variante_nombre_contenida_y_afiliacion_compartida")
                    break
        if target_gid is None:
            group_seq += 1
            target_gid = f"autor_group_{group_seq}"
            author_groups[target_gid] = {
                "variants": [], "affs": [], "indices": [], "row_positions": [],
                "exact_keys": set(), "aff_keys": set(), "merge_notes": [],
            }
            author_order.append(target_gid)

        group = author_groups[target_gid]
        group["variants"].append(raw_author)
        group["indices"].append(original_idx)
        group["row_positions"].append(row_pos)
        group["exact_keys"].add(ak)
        group["aff_keys"].update(row_aff_keys)
        if row.get("Afiliacion1"):
            group["affs"].append(row["Afiliacion1"])
        if row.get("Afiliacion2"):
            group["affs"].append(row["Afiliacion2"])

    final_rows: List[Dict[str, str]] = []
    author_diags: List[Dict[str, str]] = []
    final_authors: List[str] = []
    final_affiliations_for_article: List[str] = []

    for gid in author_order:
        ag = author_groups[gid]
        author_final = choose_author_display(ag["variants"])
        aff1, aff2, aff_candidates, aff_discarded, aff_reason = choose_affiliations(ag["affs"])
        if ag.get("merge_notes"):
            aff_reason = aff_reason + "; " + "; ".join(unique_preserve(ag["merge_notes"], key_func=lambda x: x))

        final_authors.append(author_final)
        if aff1:
            final_affiliations_for_article.append(aff1)
        if aff2:
            final_affiliations_for_article.append(aff2)

        final_rows.append({
            "indice": str(new_idx),
            "Titulo": title_final,
            "Año": year_final,
            "Autor_norm": author_final,
            "Afiliacion1": aff1,
            "Afiliacion2": aff2,
            "ISBN": isbn_final,
            "ISSN": issn_final,
            "Doi": doi_final,
            "URL": url_final,
            "Area": area_final,
            "Subarea": "",
            "Keywords": keywords_final,
            "Abstract": abstract_final,
        })

        author_diags.append({
            "indice_fusionado": str(new_idx),
            "Autor_norm_final": author_final,
            "autor_key": author_key(author_final),
            "indices_originales": " | ".join(unique_preserve(ag["indices"], key_func=lambda x: x)),
            "n_filas_originales_autor": str(len(ag["variants"])),
            "variantes_autor_originales": " | ".join(unique_preserve(ag["variants"], key_func=lambda x: x)),
            "Afiliacion1_final": aff1,
            "Afiliacion2_final": aff2,
            "afiliaciones_candidatas": " | ".join(aff_candidates),
            "afiliaciones_descartadas": " | ".join(aff_discarded),
            "motivo_fusion_autor": aff_reason,
            "Titulo_final": title_final,
            "Doi_final": doi_final,
        })

    conflicto_titulo = len({title_key(t) for t in all_titles if title_key(t)}) > 1
    conflicto_año = len(year_originales) > 1
    conflicto_area = len(area_originales) > 1
    conflicto_doi = len(doi_originales) > 1

    motivos = []
    if conflicto_titulo:
        motivos.append("Revisar título: variantes de título no idénticas; se fusionó por título parecido + año + autor compartido.")
    if conflicto_año:
        motivos.append("Revisar año: el grupo contiene más de un año.")
    if conflicto_area:
        motivos.append("Área fusionada: el artículo apareció en más de un área; se eligió una sola área final por mayoría/soporte.")
    if conflicto_doi:
        motivos.append("DOI no usado para deduplicar: el grupo contiene más de un DOI; se eligió uno como DOI final y se conservaron todos en diagnóstico.")
    if not motivos and len(cluster) > 1:
        motivos.append("Duplicado fusionado por título/año/autor sin conflicto crítico.")
    if not motivos:
        motivos.append("Artículo sin duplicado.")

    article_diag = {
        "indice_fusionado": str(new_idx),
        "indices_originales": " | ".join(cluster),
        "n_indices_originales": str(len(cluster)),
        "n_filas_originales": str(len(rows)),
        "n_filas_finales_autor": str(len(final_rows)),
        "reglas_deduplicacion": " | ".join(cluster_rules),
        "clave_principal_usada": "titulo_normalizado + año + autor_normalizado",
        "Titulo_final": title_final,
        "titulos_originales": " | ".join(unique_preserve(all_titles, key_func=lambda x: x)),
        "Año_final": year_final,
        "años_originales": " | ".join(year_originales),
        "autores_interseccion_usados": authors_used_for_merge,
        "Doi_final": doi_final,
        "dois_originales": " | ".join(doi_originales),
        "ISBN_final": isbn_final,
        "isbn_originales": " | ".join(isbn_originales),
        "ISSN_final": issn_final,
        "issn_originales": " | ".join(issn_originales),
        "URL_final": url_final,
        "urls_originales": " | ".join(unique_preserve(all_urls, key_func=lambda x: x)),
        "Area_final": area_final,
        "areas_originales": " | ".join(area_originales),
        "Keywords_final": keywords_final,
        "Abstract_longitud_final": str(len(abstract_final)),
        "n_autores_finales": str(len(final_rows)),
        "autores_finales": " | ".join(final_authors),
        "conflicto_titulo": "sí" if conflicto_titulo else "no",
        "conflicto_año": "sí" if conflicto_año else "no",
        "conflicto_area": "sí" if conflicto_area else "no",
        "conflicto_doi": "sí" if conflicto_doi else "no",
        "motivo_revision": " ".join(motivos),
    }

    article_level = {
        "indice": str(new_idx),
        "Titulo": title_final,
        "Año": year_final,
        "Autores_UNAM": " | ".join(final_authors),
        "Afiliaciones_UNAM": " | ".join(unique_preserve(final_affiliations_for_article, key_func=affiliation_key)),
        "ISBN": isbn_final,
        "ISSN": issn_final,
        "Doi": doi_final,
        "URL": url_final,
        "Area": area_final,
        "Keywords": keywords_final,
        "Abstract": abstract_final,
        "indices_originales": " | ".join(cluster),
        "n_autores_unam": str(len(final_rows)),
    }

    return final_rows, article_diag, author_diags, article_level

## Escritura y validación

In [8]:
# ---------------------------------------------------------------------
# Escritura y validación
# ---------------------------------------------------------------------
def write_csv(path: Path, rows: List[Dict[str, str]], columns: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def validate_output(rows: List[Dict[str, str]]) -> Dict[str, Any]:
    checks: Dict[str, Any] = {}
    checks["filas"] = len(rows)
    checks["columnas_canonicas_ok"] = bool(rows and list(rows[0].keys()) == CANONICAL_COLUMNS)
    checks["subarea_no_vacia"] = sum(1 for r in rows if clean_cell(r.get("Subarea", "")) != "")
    checks["doi_invalido"] = sum(1 for r in rows if r.get("Doi") and not r["Doi"].startswith("10."))
    checks["url_invalida"] = sum(1 for r in rows if r.get("URL") and not r["URL"].startswith("http"))
    checks["filas_sin_autor"] = sum(1 for r in rows if not r.get("Autor_norm"))
    checks["filas_sin_afiliacion"] = sum(1 for r in rows if not r.get("Afiliacion1") and not r.get("Afiliacion2"))
    checks["afiliaciones_genericas_unam"] = sum(
        1 for r in rows
        if is_generic_unam(r.get("Afiliacion1", "")) or is_generic_unam(r.get("Afiliacion2", ""))
    )
    checks["afiliaciones_con_coma"] = sum(
        1 for r in rows
        if "," in r.get("Afiliacion1", "") or "," in r.get("Afiliacion2", "")
    )
    checks["areas_fuera_catalogo"] = sum(1 for r in rows if r.get("Area") and r.get("Area") not in VALID_AREAS)
    seen = set()
    dup = 0
    for r in rows:
        k = (r["indice"], author_key(r["Autor_norm"]))
        if k in seen:
            dup += 1
        seen.add(k)
    checks["duplicados_indice_autor"] = dup
    return checks


def process() -> Dict[str, Any]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    input_rows = read_rows(INPUT_CSV)
    records = build_article_records(input_rows)
    clusters, cluster_rules_map, cluster_authors_map = build_clusters(records)

    final_rows: List[Dict[str, str]] = []
    article_diags: List[Dict[str, str]] = []
    author_diags: List[Dict[str, str]] = []
    article_level_rows: List[Dict[str, str]] = []

    for new_idx, cluster in enumerate(clusters, start=1):
        cluster_key = "||".join(cluster)
        cluster_rules = cluster_rules_map.get(cluster_key, ["sin_duplicado"])
        authors_used = cluster_authors_map.get(cluster_key, "")
        fused_rows, article_diag, author_diag_rows, article_level = fuse_cluster(
            cluster, new_idx, records, cluster_rules, authors_used
        )
        final_rows.extend(fused_rows)
        article_diags.append(article_diag)
        author_diags.extend(author_diag_rows)
        article_level_rows.append(article_level)

    write_csv(OUTPUT_CSV, final_rows, CANONICAL_COLUMNS)
    write_csv(OUTPUT_ARTICLES_CSV, article_level_rows, ARTICLE_LEVEL_COLUMNS)
    write_csv(DIAG_ARTICLES_CSV, article_diags, ARTICLE_DIAG_COLUMNS)
    write_csv(DIAG_AUTHORS_CSV, author_diags, AUTHOR_DIAG_COLUMNS)

    rows_original = len(input_rows)
    original_articles = len(records)
    final_articles = len(article_level_rows)
    final_author_rows = len(final_rows)
    duplicated_clusters = sum(1 for d in article_diags if int(d["n_indices_originales"]) > 1)
    original_indices_in_duplicates = sum(
        int(d["n_indices_originales"]) for d in article_diags if int(d["n_indices_originales"]) > 1
    )
    generic_before = sum(
        1 for r in input_rows
        if is_generic_unam(r.get("Afiliacion1", "")) or is_generic_unam(r.get("Afiliacion2", ""))
    )
    generic_after = sum(
        1 for r in final_rows
        if is_generic_unam(r.get("Afiliacion1", "")) or is_generic_unam(r.get("Afiliacion2", ""))
    )

    summary = {
        "input_csv": str(INPUT_CSV),
        "output_csv": str(OUTPUT_CSV),
        "output_articles_csv": str(OUTPUT_ARTICLES_CSV),
        "diagnostico_articulos_csv": str(DIAG_ARTICLES_CSV),
        "diagnostico_autores_csv": str(DIAG_AUTHORS_CSV),
        "criterio_principal_deduplicacion": "titulo_normalizado + año + autor_normalizado; DOI no se usa como clave de unión; incluye regla de año distinto 2024/2025 y equivalencias manuales de títulos revisados",
        "filas_originales_autor_articulo": rows_original,
        "articulos_originales_por_indice": original_articles,
        "articulos_finales_deduplicados": final_articles,
        "filas_finales_autor_articulo": final_author_rows,
        "filas_autor_articulo_reducidas": rows_original - final_author_rows,
        "articulos_reducidos": original_articles - final_articles,
        "clusters_con_duplicados": duplicated_clusters,
        "indices_originales_en_clusters_duplicados": original_indices_in_duplicates,
        "clusters_con_conflicto_titulo": sum(1 for d in article_diags if d["conflicto_titulo"] == "sí"),
        "clusters_con_conflicto_año": sum(1 for d in article_diags if d["conflicto_año"] == "sí"),
        "clusters_con_conflicto_area": sum(1 for d in article_diags if d["conflicto_area"] == "sí"),
        "clusters_con_conflicto_doi": sum(1 for d in article_diags if d["conflicto_doi"] == "sí"),
        "filas_autor_con_afiliacion2": sum(1 for r in final_rows if r.get("Afiliacion2")),
        "autores_unicos_finales": len({author_key(r["Autor_norm"]) for r in final_rows if author_key(r["Autor_norm"])}),
        "dois_unicos_finales_no_vacios": len({r["Doi"] for r in article_level_rows if r.get("Doi")}),
        "articulos_sin_doi_final": sum(1 for r in article_level_rows if not r.get("Doi")),
        "afiliaciones_genericas_unam_antes": generic_before,
        "afiliaciones_genericas_unam_despues": generic_after,
        "validacion_salida": validate_output(final_rows),
        "reglas_deduplicacion_clusters": dict(Counter(d["reglas_deduplicacion"] for d in article_diags)),
    }
    SUMMARY_JSON.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

if __name__ == "__main__":
    process()

{
  "input_csv": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\03_Nomralizar_Nombres\\integrado_solo_unam_autores_normalizados.csv",
  "output_csv": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\04_deduplicar\\integrado_solo_unam_deduplicado_fusionado.csv",
  "output_articles_csv": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\04_deduplicar\\integrado_solo_unam_deduplicado_fusionado_articulos.csv",
  "diagnostico_articulos_csv": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\04_deduplicar\\diagnostico_deduplicacion_fusion_articulos.csv",
  "diagnostico_autores_csv": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\04_deduplicar\\diagnostico_deduplicacion_fusion_autores.csv",
  "criterio_principal_deduplicacion": "titulo_normalizado + año + autor_normalizado; DOI no se usa como clave de unión; incluye regla de año distinto 2024/2025 y equivalencias manuales de títulos revisados",
  "filas_originales_autor_articulo": 3237,
  "articulos_originales_por_indice": 1459,
 